In [ ]:
!pip install qwen_vl_utils

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import os
from tqdm import tqdm
import json
import re
import torch
import sys

initial_prompt = """You are a meme analyzer. You should describe the meme, what is written in the meme. Finally classify the meme between HOMOPHOBIA, TRANSPHOBIA AND NON_ANTI_LGBTQ

### The text in the meme ###
<The text in the meme >

### The description of meme ###
<Analysis of the meme>

"""
def extract_steps_and_answer(response):
    """
    <Final answer>
    """
    # 定义正则表达式模式
    restate_pattern = r"### The text in the meme ###\n(.*?)\n### The description of meme ###"
    answer_pattern = r"### The description of meme ###\n(.*)"
    # classify_pattern = r"### The classification ###\n(.*)"

    # 匹配 <Restate the problem and outline the solution approach>
    restate_match = re.search(restate_pattern, response, re.DOTALL)
    ocr = restate_match.group(1).strip() if restate_match else ""

    # 匹配 <Final answer>
    answer_match = re.search(answer_pattern, response, re.DOTALL)
    analysis = answer_match.group(1).strip() if answer_match else ""

    # class_match = re.search(classify_pattern, response, re.DOTALL)
    # classification = class_match.group(1).strip() if answer_match else ""

    return ocr, analysis#, classification


from PIL import Image
import glob
def run(image_dir,model_path):
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto")
    
    processor = AutoProcessor.from_pretrained(model_path)

    # image_path = ['1.jpg',
    #              '2.jpg',
    #              '3.jpg',
    #              '4.jpg',
    #              '5.jpg',
    #              '6.jpg',
    #              '7.jpg',
    #              '8.jpg',
    #              '9.jpg',
    #              '10.jpg',]
    image_path = [str(i) + '.jpg' for i in range(139, 139+1)]
    
    res = []
    paths = [os.path.join(image_dir, filename) for filename in image_path]
    for image in tqdm(paths, desc="Processing"):
        img = Image.open(image).convert("RGB")
        w, h = img.size
        img = img.resize((int(w * 0.5), int(h * 0.5)), Image.Resampling.LANCZOS)
        # Prepare the messages for the model
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": initial_prompt},
                ],
            }
        ]

        # Preparation for inference
        
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to(model.device)

        # Inference: Generation of the output
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        ocr, analysis = extract_steps_and_answer(output_text)
        print('image：',image,'\n')
        print('Text：',ocr,'\n')
        print('Analysis：',analysis,'\n')
        # print('Classification：',classification,'\n')
        print(18*"*")
        res.append([ocr, analysis])
    return res




In [ ]:
# !cp /kaggle/input/qwen2.5-vl/transformers/3b-instruct/2 -r ./

!cp /kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/3b-instruct/2 -r ./

In [ ]:
import json

config_data = {
  "min_pixels": 3136,
  "max_pixels": 12845056,
  "patch_size": 14,
  "temporal_patch_size": 2,
  "merge_size": 2,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "image_processor_type": "Qwen2VLImageProcessor",
  "processor_class": "Qwen2_5_VLProcessor"
}

output_filename = "2/preprocessor_config.json"

with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(config_data, f, ensure_ascii=False, indent=4)

In [ ]:
!set CUDA_LAUNCH_BLOCKING = 1
!SET TORCH_USE_CUDA_DSA = 1

In [ ]:
import glob
import os
# image_dir = "/kaggle/input/math-vl/images/"
image_dir = "/kaggle/input/datasets/jishnumsc24/eng-test-im/eng_test/Test_images"
model_path = "/kaggle/working/2"
res = run(image_dir,model_path)

In [ ]:
res[0]

In [ ]:
import torch
import gc

# 1. Delete model, optimizers, and large tensors
del model
del optimizer
# del other_large_variables

# 2. Force Python garbage collection
gc.collect()

# 3. Clear PyTorch's cache
torch.cuda.empty_cache()


In [ ]:
# import pandas as pd
# submission = pd.read_csv('/kaggle/input/math-vl/submission.csv')
# submission['answer'] = res
# submission.to_csv('submission.csv',index=False)
import pandas as pd
df = pd.DataFrame(res, columns=["ocr", "analysis", "pred"])

# Save to CSV
df.to_csv("output_2.csv", index=False)

In [ ]:
# submission